# Model Evaluation — Agentic Workflow Testing

This notebook runs a standardized set of agentic coding tasks through Claude Code, comparing a self-hosted model against a baseline.

## How It Works

1. Defines 15 tasks across 3 difficulty tiers (basic, intermediate, complex)
2. Runs each task via `claude -p` (non-interactive mode) against the **baseline** (your existing Claude Code setup)
3. Runs the same tasks against the **self-hosted model** (vLLM on RHOAI)
4. Captures structured output (response, tokens, cost) for each task
5. Generates a side-by-side comparison table

## Evaluating Multiple Models

The baseline only needs to run once. On subsequent runs, the notebook auto-detects the saved baseline in `results/` and loads it instead of re-running. Just change `KSERVE_ENDPOINT` and `MODEL_NAME` to evaluate a different model.

## Prerequisites

- **Claude Code installed, authenticated, and working** — the baseline run uses your existing Claude Code setup as-is (Vertex AI, personal account, API key — any backend works). Verify with: `claude -p "What is 2 + 2?"`
- A self-hosted model deployed via vLLM (see [deploy-model.md](../setup/deploy-model.md))
- `oc` CLI logged in (if running against KServe)

---

## Configuration

Fill in your deployment details and API key below.

In [ ]:
# --- Self-hosted model config ---
KSERVE_ENDPOINT = "<your-endpoint>"          # e.g. http://localhost:8000 or https://qwen3-coder-my-namespace.apps.my-cluster.openshiftapps.com
MODEL_NAME = "qwen3-coder"                  # must match --served-model-name

# --- Baseline config ---
BASELINE_MODEL = "claude-sonnet-4-6"         # baseline model for comparison
BASELINE_RESULTS_FILE = f"results/baseline-{BASELINE_MODEL}.json"

# --- Test settings ---
MAX_TURNS = 30                               # max agentic turns per task
TIMEOUT = 600                                # seconds before a task is killed

print(f"Using self-hosted model '{MODEL_NAME}'")
print(f"Baseline model for comparison: '{BASELINE_MODEL}'")

---

## Helper Functions

In [57]:
import json, os, subprocess, shutil, time, tempfile
from pathlib import Path
from datetime import datetime

BASE_DIR = Path(tempfile.mkdtemp(prefix="claude-eval-"))
print(f"Working directory: {BASE_DIR}")


def setup_workdir(backend, group):
    """Create a clean working directory for a (backend, group) pair."""
    workdir = BASE_DIR / backend / group
    workdir.mkdir(parents=True, exist_ok=True)
    return workdir


def write_fixture(workdir, filename, content):
    """Write a test fixture file into the working directory."""
    (workdir / filename).write_text(content)


def run_task(task, env, workdir):
    """Run a single task via claude -p and return structured results."""
    cmd = [
        "claude", "-p", task["prompt"],
        "--output-format", "json",
        "--max-turns", str(MAX_TURNS),
        "--allowedTools", "Read,Edit,Write,Bash",
    ]

    start = time.time()
    try:
        result = subprocess.run(
            cmd, capture_output=True, text=True,
            cwd=str(workdir), env=env, timeout=TIMEOUT,
            stdin=subprocess.DEVNULL,
        )
        elapsed = round(time.time() - start, 1)

        # Always try to parse JSON output — even on non-zero return code,
        # claude -p still writes structured JSON (e.g. error_max_turns).
        # Only fall back to the error path if there's no parseable stdout.
        if not result.stdout.strip():
            return {
                "task_id": task["id"],
                "name": task["name"],
                "status": "error",
                "error": result.stderr[:500] if result.stderr else "No output from claude -p",
                "elapsed_s": elapsed,
            }

        # claude -p --output-format json returns either:
        # - A JSON array: [{"type":"system",...}, {"type":"result",...}]  (Anthropic API / Vertex)
        # - A single JSON dict: {"type":"result",...}  (custom endpoints like vLLM)
        # On very long sessions, the JSON output can be truncated — handle parse errors gracefully.
        try:
            parsed = json.loads(result.stdout)
        except json.JSONDecodeError:
            return {
                "task_id": task["id"],
                "name": task["name"],
                "status": "error",
                "error": f"JSON output truncated ({len(result.stdout)} chars)",
                "elapsed_s": elapsed,
            }

        if isinstance(parsed, list):
            result_item = next(
                (item for item in parsed if isinstance(item, dict) and item.get("type") == "result"),
                {},
            )
        elif isinstance(parsed, dict):
            result_item = parsed if parsed.get("type") == "result" else {}
        else:
            result_item = {}

        subtype = result_item.get("subtype", "")
        if subtype == "success":
            status = "completed"
        elif subtype == "error_max_turns":
            status = "max_turns"
        else:
            status = "error"

        return {
            "task_id": task["id"],
            "name": task["name"],
            "status": status,
            "result": result_item.get("result", ""),
            "session_id": result_item.get("session_id", ""),
            "input_tokens": result_item.get("usage", {}).get("input_tokens", 0),
            "output_tokens": result_item.get("usage", {}).get("output_tokens", 0),
            "cost_usd": result_item.get("total_cost_usd", 0),
            "num_turns": result_item.get("num_turns", 0),
            "elapsed_s": elapsed,
        }

    except subprocess.TimeoutExpired:
        return {
            "task_id": task["id"],
            "name": task["name"],
            "status": "timeout",
            "elapsed_s": TIMEOUT,
        }
    except Exception as e:
        return {
            "task_id": task["id"],
            "name": task["name"],
            "status": "error",
            "error": str(e),
            "elapsed_s": round(time.time() - start, 1),
        }

Working directory: /var/folders/9n/h8fx6zrj7_797rg65n2tv1gw0000gn/T/claude-eval-7t4ef_zy


---

## Task Definitions

Tasks are grouped so that dependent tasks share a working directory. Within each group, tasks run sequentially.

In [58]:
TASKS = [
    # --- Basic (Group 1) ---
    {
        "id": 1, "group": "basic", "name": "Simple prompt",
        "prompt": "What is 2 + 2?",
        "pass_criteria": "Correct answer returned",
    },
    {
        "id": 2, "group": "basic", "name": "File creation",
        "prompt": "Create a file hello.py that prints 'Hello, World!'",
        "pass_criteria": "File exists, runs correctly",
    },
    {
        "id": 3, "group": "basic", "name": "File read + analysis",
        "prompt": "Read hello.py and count the lines",
        "pass_criteria": "Reads file, reports accurate count",
    },
    {
        "id": 4, "group": "basic", "name": "Code generation",
        "prompt": "Write fibonacci.py that takes n from the command line and prints the first n Fibonacci numbers. Include error handling for invalid input.",
        "pass_criteria": "File runs for n=10, rejects invalid input",
    },
    {
        "id": 5, "group": "basic", "name": "File editing",
        "prompt": "Add type hints and docstrings to fibonacci.py",
        "pass_criteria": "Edits in place, adds type hints, doesn't break existing logic",
    },
    {
        "id": 6, "group": "basic", "name": "Bash tool",
        "prompt": "Run fibonacci.py with n=10",
        "pass_criteria": "Executes script, correct output",
    },

    # --- Intermediate (Group 2) ---
    {
        "id": 7, "group": "intermediate", "name": "Multi-file project",
        "prompt": "Create a calculator module (calculator.py) with add, subtract, multiply, divide functions. Then create test_calculator.py with pytest tests for each function. Run the tests.",
        "pass_criteria": "Both files created, pytest runs, all tests pass",
    },
    {
        "id": 8, "group": "intermediate", "name": "Bug fixing",
        "prompt": "Read buggy_sort.py and find the bug. Fix it and verify by running the script.",
        "pass_criteria": "Identifies bug, fixes it, script produces sorted output",
        "fixture": {
            "filename": "buggy_sort.py",
            "content": "def bubble_sort(arr):\n    n = len(arr)\n    for i in range(n):\n        for j in range(0, n - i - 1):\n            if arr[j] > arr[j + 1]:\n                arr[j] = arr[j]  # Bug: should swap arr[j] and arr[j+1]\n    return arr\n\n\ndata = [64, 34, 25, 12, 22, 11, 90]\nprint(f'Unsorted: {data}')\nprint(f'Sorted:   {bubble_sort(data)}')\n",
        },
    },
    {
        "id": 9, "group": "intermediate", "name": "Code explanation",
        "prompt": "Read calculator.py and explain how each function works",
        "pass_criteria": "Reads file, explanation is accurate and covers all functions",
    },
    {
        "id": 10, "group": "intermediate", "name": "Test-driven fix",
        "prompt": "The tests in test_calculator.py are failing. Investigate why, fix the issue, and re-run the tests until they pass.",
        "pass_criteria": "Reads test output, identifies root cause, fixes code, all tests pass",
        "fixture": {
            "filename": "test_calculator.py",
            "content": "import pytest\nfrom calculator import add, subtract, multiply, divide\n\n\ndef test_add():\n    assert add(2, 3) == 5\n\ndef test_subtract():\n    assert subtract(10, 4) == 6\n\ndef test_multiply():\n    assert multiply(3, 7) == 21\n\ndef test_divide():\n    assert divide(10, 2) == 5.0\n\ndef test_divide_by_zero():\n    with pytest.raises(ValueError):\n        divide(10, 0)\n",
        },
    },

    # --- Complex (Group 3) ---
    {
        "id": 11, "group": "complex", "name": "REST API scaffolding",
        "prompt": "Create a Flask REST API with CRUD endpoints for a todo list. Include a Todo model, routes, error handling, and a requirements.txt.",
        "pass_criteria": "All files created, app starts without errors, endpoints respond to curl",
    },
    {
        "id": 12, "group": "complex", "name": "Multi-file refactoring",
        "prompt": "Refactor the Flask app into a proper project structure — separate files for models, routes, and config. Update all imports.",
        "pass_criteria": "Files restructured, imports updated, app still starts and endpoints work",
    },
    {
        "id": 13, "group": "complex", "name": "Debugging with context",
        "prompt": "The /todos POST endpoint returns 500. Debug by reading the code, identify the issue, fix it, and verify with a curl command.",
        "pass_criteria": "Reads code, identifies root cause, fixes it, curl returns 201",
    },
    {
        "id": 14, "group": "complex", "name": "Long multi-turn session",
        "prompt": "Add pagination to the GET /todos endpoint, write tests for it, then add filtering by completion status. Run all tests after each change.",
        "pass_criteria": "Pagination works, filtering works, tests pass after each step",
    },
    {
        "id": 15, "group": "complex", "name": "Cross-file analysis",
        "prompt": "Find all functions across the project that don't have error handling. Add try/except blocks where appropriate.",
        "pass_criteria": "Scans multiple files, adds error handling where missing, doesn't break existing tests",
    },
]

print(f"Loaded {len(TASKS)} tasks across {len(set(t['group'] for t in TASKS))} groups")

Loaded 15 tasks across 3 groups


---

## Environment Setup

The baseline uses your existing Claude Code auth (whatever is configured in `~/.claude`). The self-hosted model uses `oc whoami -t` (local) or the service account token (workbench).

In [59]:
# --- Baseline environment (uses existing Claude Code auth) ---
# Inherits the current environment as-is — works with Vertex AI, personal API key, or any other setup.
# Only strips self-hosted model vars so they don't interfere with the baseline.
baseline_env = os.environ.copy()
baseline_env.pop("ANTHROPIC_BASE_URL", None)
baseline_env.pop("ANTHROPIC_AUTH_TOKEN", None)
baseline_env.pop("ANTHROPIC_CUSTOM_MODEL_OPTION", None)
for key in list(baseline_env):
    if key.startswith("ANTHROPIC_DEFAULT_"):
        baseline_env.pop(key)

# --- Self-hosted model environment ---
SA_TOKEN_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/token"

if os.path.exists(SA_TOKEN_PATH):
    with open(SA_TOKEN_PATH) as f:
        auth_token = f.read().strip()
    print("Using service account token (workbench)")
else:
    result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    auth_token = result.stdout.strip()
    if not auth_token:
        print("Warning: 'oc whoami -t' failed — model run will fail. Log into oc first.")
    else:
        print("Using oc token (local)")

model_env = os.environ.copy()
model_env["ANTHROPIC_BASE_URL"] = KSERVE_ENDPOINT
model_env["ANTHROPIC_AUTH_TOKEN"] = auth_token
model_env["CLAUDE_CODE_SKIP_AUTH_LOGIN"] = "1"
model_env["NODE_TLS_REJECT_UNAUTHORIZED"] = "0"
model_env["CLAUDE_CONFIG_DIR"] = str(BASE_DIR / "config-model")
model_env["ANTHROPIC_DEFAULT_OPUS_MODEL"] = MODEL_NAME
model_env["ANTHROPIC_DEFAULT_SONNET_MODEL"] = MODEL_NAME
model_env["ANTHROPIC_DEFAULT_HAIKU_MODEL"] = MODEL_NAME
model_env["ANTHROPIC_CUSTOM_MODEL_OPTION"] = MODEL_NAME
model_env["CLAUDE_CODE_USE_VERTEX"] = "0"
model_env.pop("ANTHROPIC_API_KEY", None)

print("Environments configured.")

Using oc token (local)
Environments configured.


---

## Run Tasks

The cell below defines a runner function that executes all tasks for a given backend. Tasks within a group run sequentially (they depend on each other); groups are independent.

In [60]:
def save_incremental(results, output_file):
    """Save results to JSON after each task."""
    Path("results").mkdir(exist_ok=True)
    Path(output_file).write_text(json.dumps(results, indent=2))


def run_all_tasks(backend_name, env, output_file):
    """Run all tasks for a backend, saving results after each task."""
    results = []
    groups = {}
    for task in TASKS:
        groups.setdefault(task["group"], []).append(task)

    for group_name, group_tasks in groups.items():
        workdir = setup_workdir(backend_name, group_name)
        print(f"\n{'='*60}")
        print(f"Group: {group_name} | Backend: {backend_name}")
        print(f"Workdir: {workdir}")
        print(f"{'='*60}")

        for task in group_tasks:
            if "fixture" in task:
                write_fixture(workdir, task["fixture"]["filename"], task["fixture"]["content"])
                print(f"  Wrote fixture: {task['fixture']['filename']}")

            print(f"\n  Task {task['id']}: {task['name']}...")
            result = run_task(task, env, workdir)
            results.append(result)

            status = result["status"]
            elapsed = result["elapsed_s"]
            tokens = result.get("input_tokens", 0) + result.get("output_tokens", 0)
            print(f"  Status: {status} | Time: {elapsed}s | Tokens: {tokens}")

            if status == "error":
                print(f"  Error: {result.get('error', 'unknown')[:200]}")

            save_incremental(results, output_file)

    return results

---

## Baseline Run (Claude Sonnet 4)

If `BASELINE_RESULTS_FILE` is set in the config cell, the baseline is loaded from that file instead of re-running. This saves time and cost when evaluating multiple models — run the baseline once, reuse it for every model.

In [61]:
if Path(BASELINE_RESULTS_FILE).exists():
    with open(BASELINE_RESULTS_FILE) as f:
        saved = json.load(f)
    baseline_results = saved["baseline_results"]
    print(f"Loaded baseline from {BASELINE_RESULTS_FILE}")
    print(f"Baseline model: {saved['baseline_model']}")
    print(f"Tasks loaded: {len(baseline_results)}")
else:
    print(f"No baseline found at {BASELINE_RESULTS_FILE} — running fresh ({BASELINE_MODEL})...")
    print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    baseline_results = run_all_tasks("baseline", baseline_env, BASELINE_RESULTS_FILE)

    # Wrap with metadata and save final version
    baseline_output = {
        "date": datetime.now().strftime("%Y-%m-%d"),
        "baseline_model": BASELINE_MODEL,
        "baseline_results": baseline_results,
    }
    Path(BASELINE_RESULTS_FILE).write_text(json.dumps(baseline_output, indent=2))
    print(f"\n\nBaseline run complete — {len(baseline_results)} tasks finished.")

No baseline found at results/baseline-claude-sonnet-4-6.json — running fresh (claude-sonnet-4-6)...
Time: 2026-05-19 17:14:58


Group: basic | Backend: baseline
Workdir: /var/folders/9n/h8fx6zrj7_797rg65n2tv1gw0000gn/T/claude-eval-7t4ef_zy/baseline/basic

  Task 1: Simple prompt...
  Status: completed | Time: 2.1s | Tokens: 8

  Task 2: File creation...
  Status: completed | Time: 7.7s | Tokens: 154

  Task 3: File read + analysis...
  Status: completed | Time: 6.2s | Tokens: 142

  Task 4: Code generation...
  Status: completed | Time: 14.1s | Tokens: 593

  Task 5: File editing...
  Status: completed | Time: 14.9s | Tokens: 811

  Task 6: Bash tool...
  Status: completed | Time: 9.4s | Tokens: 371

Group: intermediate | Backend: baseline
Workdir: /var/folders/9n/h8fx6zrj7_797rg65n2tv1gw0000gn/T/claude-eval-7t4ef_zy/baseline/intermediate

  Task 7: Multi-file project...
  Status: completed | Time: 19.8s | Tokens: 1002
  Wrote fixture: buggy_sort.py

  Task 8: Bug fixing...
  Status: c

---

## Model Run

Run the same 15 tasks against the self-hosted model.

In [62]:
MODEL_RESULTS_FILE = f"results/{MODEL_NAME}-{datetime.now().strftime('%Y-%m-%d')}.json"

print(f"Starting model run ({MODEL_NAME})...")
print(f"Results saving to: {MODEL_RESULTS_FILE}")
print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

model_results = run_all_tasks("model", model_env, MODEL_RESULTS_FILE)

print(f"\n\nModel run complete — {len(model_results)} tasks finished.")

Starting model run (qwen3-coder)...
Results saving to: results/qwen3-coder-2026-05-19.json
Time: 2026-05-19 17:26:04


Group: basic | Backend: model
Workdir: /var/folders/9n/h8fx6zrj7_797rg65n2tv1gw0000gn/T/claude-eval-7t4ef_zy/model/basic

  Task 1: Simple prompt...
  Status: completed | Time: 5.1s | Tokens: 24694

  Task 2: File creation...
  Status: completed | Time: 6.4s | Tokens: 24789

  Task 3: File read + analysis...
  Status: completed | Time: 11.8s | Tokens: 49725

  Task 4: Code generation...
  Status: completed | Time: 29.4s | Tokens: 100975

  Task 5: File editing...
  Status: completed | Time: 31.7s | Tokens: 101522

  Task 6: Bash tool...
  Status: completed | Time: 21.6s | Tokens: 100334

Group: intermediate | Backend: model
Workdir: /var/folders/9n/h8fx6zrj7_797rg65n2tv1gw0000gn/T/claude-eval-7t4ef_zy/model/intermediate

  Task 7: Multi-file project...
  Status: completed | Time: 43.5s | Tokens: 155313
  Wrote fixture: buggy_sort.py

  Task 8: Bug fixing...
  Status: c

---

## Results Comparison

In [ ]:
# Build comparison table
header = f"{'#':<4} {'Task':<28} {'Baseline':<12} {'Model':<12} {'Base Tokens':<14} {'Model Tokens':<14} {'Base Time':<12} {'Model Time':<12}"
separator = "-" * len(header)

print(f"\n{'Results Comparison':^{len(header)}}")
print(f"{BASELINE_MODEL} vs {MODEL_NAME}")
print(separator)
print(header)
print(separator)

for b, m in zip(baseline_results, model_results):
    b_tokens = b.get("input_tokens", 0) + b.get("output_tokens", 0)
    m_tokens = m.get("input_tokens", 0) + m.get("output_tokens", 0)
    print(
        f"{b['task_id']:<4} {b['name']:<28} {b['status']:<12} {m['status']:<12} "
        f"{b_tokens:<14} {m_tokens:<14} {b['elapsed_s']:<12} {m['elapsed_s']:<12}"
    )

print(separator)

# Totals
b_total_tokens = sum(r.get("input_tokens", 0) + r.get("output_tokens", 0) for r in baseline_results)
m_total_tokens = sum(r.get("input_tokens", 0) + r.get("output_tokens", 0) for r in model_results)
b_total_cost = sum(r.get("cost_usd", 0) for r in baseline_results)
b_total_time = sum(r.get("elapsed_s", 0) for r in baseline_results)
m_total_time = sum(r.get("elapsed_s", 0) for r in model_results)

print(f"\nBaseline total: {b_total_tokens} tokens | ${b_total_cost:.4f} | {b_total_time:.1f}s")
print(f"Model total:    {m_total_tokens} tokens | {m_total_time:.1f}s")

---

## Save Final Results

Results are saved incrementally after each task. This cell writes the final combined file with metadata.

In [64]:
# Write final combined results with full metadata
final_output = {
    "date": datetime.now().strftime("%Y-%m-%d"),
    "baseline_model": BASELINE_MODEL,
    "self_hosted_model": MODEL_NAME,
    "kserve_endpoint": KSERVE_ENDPOINT,
    "max_turns": MAX_TURNS,
    "timeout_s": TIMEOUT,
    "baseline_results": baseline_results,
    "model_results": model_results,
}
Path(MODEL_RESULTS_FILE).write_text(json.dumps(final_output, indent=2))
print(f"Final results saved to {MODEL_RESULTS_FILE}")

Final results saved to results/qwen3-coder-2026-05-19.json


---

## Inspect Individual Results

Use this cell to view the full response for any task. Change the `task_id` and `backend` variables.

In [65]:
# --- Change these to inspect a specific result ---
task_id = 1
backend = "baseline"  # "baseline" or "model"

results = baseline_results if backend == "baseline" else model_results
match = next((r for r in results if r["task_id"] == task_id), None)

if match:
    print(f"Task {task_id} ({match['name']}) — {backend}")
    print(f"Status: {match['status']}")
    print(f"Tokens: {match.get('input_tokens', 0) + match.get('output_tokens', 0)}")
    print(f"Time: {match['elapsed_s']}s")
    print(f"\n{'─' * 60}\n")
    print(match.get("result", match.get("error", "No output")))
else:
    print(f"No result found for task {task_id}")

Task 1 (Simple prompt) — baseline
Status: completed
Tokens: 8
Time: 2.1s

────────────────────────────────────────────────────────────

4


---

## Next Steps

1. Review the comparison table above and the full responses for each task
2. Fill in the evaluation template at `model-evaluation-template.md` with your assessment (Pass/Partial/Fail per task)
3. Document observed limitations and differences in the template's summary sections
4. To evaluate a different model, update `KSERVE_ENDPOINT` and `MODEL_NAME` in the config cell and re-run